# Saccade classification and behavioral measures

This notebook calculates behavioral measures from manually reviewed saccade annotations. It produces participant-level summaries, predictors for the statistical analyses, and descriptive statistics for Table 2.

## How to run

1. Set `SACCADES_DIR` to the folder containing the reviewed Excel files from `00_saccade_detection.ipynb`. Save each reviewed file as `<participant_id>_saccades.xlsx`.
2. Check `PARTICIPANT_IDS` and `EXPECTED_N_PARTICIPANTS`. The configured list defines the 69-participant study cohort.
3. Set `RUN_STUDY_ANALYSIS = True`, restart the kernel, and run all cells in order.
4. Inspect the loading summary, quality-control tables, participant summaries, and Table 2.
5. To save the results, set `OUTPUT_DIR` and `EXPORT_OUTPUTS = True`, then rerun the export cell.

With `RUN_STUDY_ANALYSIS = False`, only the synthetic classification examples run. Participant data are required to calculate the study results. The input workbooks are read without modification.

## Dependencies

Python with `numpy`, `pandas`, `openpyxl`, and a Jupyter environment. Package versions are displayed and included in the exported summary workbook.

## Reviewed input files

One file per participant: `<participant_id>_saccades.xlsx`. The first worksheet must contain one row per target trial, including trials without a detected saccade.

| Required column | Meaning |
|---|---|
| `trial` | Nonnegative integer identifying the retained target epoch; unique within a participant |
| `event` | Target event code: 20, 21, 22, 23, 30, 31, 32, or 33 |
| `latency_ms` | Reviewed saccade onset relative to target onset, in milliseconds; negative values are allowed; missing for an undetected saccade |
| `direction` | Reviewed direction: −1 = left, +1 = right; 0 or missing for an undetected saccade |

All analysis variables are derived from `event`, `latency_ms`, and `direction`. Additional annotation columns do not determine the analysis. Task descriptors are assigned from the event dictionary, and detection, direction correctness, and latency categories are calculated again from the reviewed values.

Completely empty rows and unnamed Excel index columns are removed. Invalid records stop processing with a participant and file reference. The input trial identifiers and row order are preserved.

The `trial` identifier is an epoch index, not an EEG sample index. Use the event-alignment information exported by `00_saccade_detection.ipynb` when linking annotations to EEG trials.

## Definitions

| Measure or flag | Definition |
|---|---|
| Detected saccade | Finite latency and direction −1 or +1 |
| Anticipatory | Detected saccade with latency < 90 ms, including negative latencies |
| Express | Detected saccade with 90 ≤ latency < 120 ms |
| Regular | Detected saccade with latency ≥ 120 ms |
| Response latency | Latency of a correctly directed express or regular saccade; no additional upper cutoff |
| Anticipatory percentage | 100 × anticipatory count / detected count |
| Direction-error percentage | 100 × wrong-direction count / detected count, irrespective of latency |
| Pretarget | Detected saccade with latency < 0 ms |
| `cnv_saccade_ok` | Detected saccade with latency ≥ 0 ms, irrespective of direction correctness |

The CNV flag represents the saccade timing criterion. EEG artifact rejection, available epochs, and participant-level criteria are applied separately. A latency of 0–89 ms is anticipatory for the behavioral analysis and passes this CNV timing criterion.

Percentages are missing when no saccades were detected. No intermediate rounding is applied.

For the anticipatory probability contrast, valid and invalid counts are pooled within each context before calculating percentages:

$$D_{\mathrm{ant}} = 100\frac{n_{\mathrm{ant},80}}{n_{\mathrm{detected},80}} - 100\frac{n_{\mathrm{ant},50}}{n_{\mathrm{detected},50}}.$$

The contrast is expressed in percentage points. The separate `antic_mean` predictor is the equal-weight mean of the available condition-level anticipatory percentages.

In [1]:
from pathlib import Path
from importlib.metadata import version
import platform
import hashlib

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 15)

software_versions = pd.DataFrame(
    [("Python", platform.python_version())]
    + [(package, version(package)) for package in ("numpy", "pandas", "openpyxl")],
    columns=["software", "version"],
)
display(software_versions)

,software,version
0,Python,3.12.14
1,numpy,2.3.5
2,pandas,2.2.3
3,openpyxl,3.1.5


In [2]:
# Preview mode runs definitions and synthetic checks without accessing study data.
RUN_STUDY_ANALYSIS = False

SACCADES_DIR = Path("PATH_TO_REVIEWED_SACCADES").expanduser()
OUTPUT_DIR = Path("PATH_TO_PRIVATE_OUTPUTS").expanduser()
EXPORT_OUTPUTS = False
OVERWRITE_OUTPUTS = False

EXPECTED_N_PARTICIPANTS = 69

# Pseudonymous participant identifiers defining the study cohort.
PARTICIPANT_IDS = """
R01 R02 R03 R06 R08 R09 R10 R12 R13
R14 R15 R16 R17 R20 R21 R22 R24 R25
R26 R27 R28 R29 R30 R31 R32 R33 R34
R35 R37 R38 R39 R40 R41 S01 S07 S09
S10 S12 S13 S20 S22 S27 S28 S29 S30
S35 S38 S43 S44 S45 S47 S49 Z01 Z02
Z03 Z04 Z05 Z06 Z07 Z09 Z10 Z11 Z12
Z13 Z15 Z16 Z17 Z19 Z20
""".split()

PRETARGET_CUTOFF_MS = 0.0
ANTICIPATORY_CUTOFF_MS = 90.0
REGULAR_CUTOFF_MS = 120.0

CONDITION_ORDER = ["valid_80", "invalid_80", "valid_50", "invalid_50"]

if EXPORT_OUTPUTS and not RUN_STUDY_ANALYSIS:
    raise ValueError("Enable RUN_STUDY_ANALYSIS before requesting export.")

print("Mode:", "study analysis" if RUN_STUDY_ANALYSIS else "preview; no study data loaded")

Mode: preview; no study data loaded


## Event coding

Cue and target directions are represented separately: −1 = left and +1 = right. Direction correctness is defined relative to the target. For events 21 and 23, the cue is right and the target is left; for events 31 and 33, the cue is left and the target is right.

Condition labels combine validity and probability: `valid_80`, `invalid_80`, `valid_50`, and `invalid_50`.

In [3]:
EVENT_INFO = pd.DataFrame(
    [
        (20, 80, "valid",   -1, -1),
        (21, 80, "invalid",  1, -1),
        (22, 50, "valid",   -1, -1),
        (23, 50, "invalid",  1, -1),
        (30, 80, "valid",    1,  1),
        (31, 80, "invalid", -1,  1),
        (32, 50, "valid",    1,  1),
        (33, 50, "invalid", -1,  1),
    ],
    columns=["event", "probability", "validity", "cue_direction", "target_direction"],
).set_index("event")
EVENT_INFO["task_condition"] = (
    EVENT_INFO["validity"] + "_" + EVENT_INFO["probability"].astype(str)
)
display(EVENT_INFO)

,probability,validity,cue_direction,target_direction,task_condition
event,,,,,
20,80,valid,-1,-1,valid_80
21,80,invalid,1,-1,invalid_80
22,50,valid,-1,-1,valid_50
23,50,invalid,1,-1,invalid_50
30,80,valid,1,1,valid_80
31,80,invalid,-1,1,invalid_80
32,50,valid,1,1,valid_50
33,50,invalid,-1,1,invalid_50


## Derive trial-level variables

The function creates an analysis copy of the input table. Task descriptors are mapped from the event code. Latency categories and direction correctness are derived from the reviewed latency and direction. Inconsistent latency/direction pairs stop processing and identify the affected rows.

In [4]:
def add_saccade_features(table):
    """Derive analysis variables from manually reviewed latency and direction."""
    missing = {"event", "latency_ms", "direction"} - set(table.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    out = table.copy()
    event = pd.to_numeric(out["event"], errors="raise")
    latency = pd.to_numeric(out["latency_ms"], errors="raise").astype(float)
    direction = pd.to_numeric(out["direction"], errors="raise").astype(float)

    if not event.isin(EVENT_INFO.index).all():
        raise ValueError("Missing or unknown target event codes.")
    if np.isinf(latency).any():
        raise ValueError("Infinite saccade latencies are not valid.")
    if not (direction.isna() | direction.isin([-1, 0, 1])).all():
        raise ValueError("Direction must be -1, 0, +1, or missing.")

    has_latency = np.isfinite(latency)
    has_direction = direction.isin([-1, 1])
    inconsistent = has_latency != has_direction
    if inconsistent.any():
        row_labels = out.loc[inconsistent, "source_excel_row"].tolist() if "source_excel_row" in out else out.index[inconsistent].tolist()
        raise ValueError(f"Latency/direction mismatch at rows {row_labels[:10]}; inspect reviewed annotations.")

    for column in EVENT_INFO.columns:
        out[column] = event.map(EVENT_INFO[column])

    detected = has_latency & has_direction
    out["saccade_detected"] = detected
    out["is_pretarget"] = detected & (latency < PRETARGET_CUTOFF_MS)
    out["is_anticipatory"] = detected & (latency < ANTICIPATORY_CUTOFF_MS)
    out["is_express"] = detected & (latency >= ANTICIPATORY_CUTOFF_MS) & (latency < REGULAR_CUTOFF_MS)
    out["is_regular"] = detected & (latency >= REGULAR_CUTOFF_MS)
    out["cnv_saccade_ok"] = detected & (latency >= PRETARGET_CUTOFF_MS)

    out["is_direction_correct"] = direction.eq(out["target_direction"]).astype("boolean").where(detected, pd.NA)
    out["is_direction_error"] = detected & direction.ne(out["target_direction"])
    rt_ok = detected & (latency >= ANTICIPATORY_CUTOFF_MS) & out["is_direction_correct"].fillna(False)
    out["rt_ms"] = latency.where(rt_ok)
    out["behavioral_class"] = np.select(
        [out["is_anticipatory"], out["is_express"], out["is_regular"]],
        ["anticipatory", "express", "regular"],
        default="not_detected",
    )
    return out

## Classification examples

These synthetic examples illustrate the classification boundaries. The final example is an incorrectly directed 100-ms saccade: it passes the CNV timing criterion and is excluded from the response-latency measure.

In [5]:
example = pd.DataFrame({
    "event": [20, 20, 20, 20, 20, 20, 20, 31],
    "latency_ms": [-20, 0, 89, 90, 119.5, 120, np.nan, 100],
    "direction": [-1, -1, -1, -1, -1, -1, 0, -1],
})
example_before = example.copy(deep=True)
checked = add_saccade_features(example)
assert checked["behavioral_class"].tolist() == [
    "anticipatory", "anticipatory", "anticipatory", "express",
    "express", "regular", "not_detected", "express",
]
assert checked["cnv_saccade_ok"].tolist() == [False, True, True, True, True, True, False, True]
assert checked.loc[7, "is_direction_error"] and pd.isna(checked.loc[7, "rt_ms"])
pd.testing.assert_frame_equal(example, example_before)
display(checked[["event", "latency_ms", "behavioral_class", "is_pretarget", "cnv_saccade_ok", "is_direction_correct", "rt_ms"]])
print("Synthetic classification checks passed.")

Synthetic classification checks passed.


,event,latency_ms,behavioral_class,is_pretarget,cnv_saccade_ok,is_direction_correct,rt_ms
0,20,-20.0,anticipatory,True,False,True,NaN
1,20,0.0,anticipatory,False,True,True,NaN
2,20,89.0,anticipatory,False,True,True,NaN
3,20,90.0,express,False,True,True,90.0
4,20,119.5,express,False,True,True,119.5
5,20,120.0,regular,False,True,True,120.0
6,20,NaN,not_detected,False,False,<NA>,NaN
7,31,100.0,express,False,True,False,NaN


## Load the specified cohort

Files are matched to the configured participant identifiers without regard to letter case. Duplicate files, missing participants, duplicate trial identifiers, unknown event codes, and absent experimental conditions stop processing. Files outside the specified cohort are listed separately.

Trials without a detected saccade remain in the data. Input Excel row numbers and SHA-256 file hashes are recorded to identify the reviewed annotations used in the analysis.

In [6]:
def load_reviewed_saccades(directory, participant_ids, expected_n):
    """Load the specified cohort without modifying or silently skipping inputs."""
    directory = Path(directory)
    ids = [str(subject).strip().upper() for subject in participant_ids]
    if len(ids) != expected_n or len(ids) != len(set(ids)) or any(not subject for subject in ids):
        raise ValueError("Cohort IDs must be nonempty, unique, and match the expected sample size.")
    if not directory.is_dir():
        raise FileNotFoundError(f"Set SACCADES_DIR to the reviewed input folder: {directory}")

    file_map = {}
    suffix = "_saccades.xlsx"
    for file in sorted(directory.iterdir()):
        if not file.is_file() or file.name.startswith("~$") or not file.name.lower().endswith(suffix):
            continue
        subject = file.name[:-len(suffix)].strip().upper()
        if subject in file_map:
            raise ValueError(f"Duplicate saccade files for participant {subject}.")
        file_map[subject] = file
    missing_files = sorted(set(ids) - set(file_map))
    if missing_files:
        raise FileNotFoundError(f"Missing cohort files: {missing_files}")
    extra_files = pd.DataFrame(
        [(subject, file_map[subject].name) for subject in sorted(set(file_map) - set(ids))],
        columns=["subject", "excluded_file_outside_cohort"],
    )

    parts, log = [], []
    for subject in ids:
        file = file_map[subject]
        try:
            table = pd.read_excel(file, sheet_name=0, engine="openpyxl")
            table.columns = table.columns.astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False)
            if table.columns.duplicated().any():
                raise ValueError("Duplicate column names after normalization.")
            table = table.loc[:, ~table.columns.str.startswith("unnamed:")].copy()
            missing = {"trial", "event", "latency_ms", "direction"} - set(table.columns)
            if missing:
                raise ValueError(f"Missing required columns: {sorted(missing)}")
            reserved = {"subject", "source_file", "source_excel_row"} & set(table.columns)
            if reserved:
                raise ValueError(f"Input already contains reserved provenance fields: {sorted(reserved)}")

            # Treat whitespace-only cells as empty for the blank-row check.
            blank_rows = table.replace(r"^\s*$", np.nan, regex=True).isna().all(axis=1)
            n_rows_read = len(table)
            excel_rows = pd.Series(np.arange(2, n_rows_read + 2), index=table.index)
            table = table.loc[~blank_rows].copy()
            table["source_excel_row"] = excel_rows.loc[table.index]
            if table.empty:
                raise ValueError("No target trials were found.")

            trial_ids = pd.to_numeric(table["trial"], errors="raise").astype(float)
            if not (np.isfinite(trial_ids) & trial_ids.ge(0) & trial_ids.eq(np.floor(trial_ids))).all():
                raise ValueError("Trial IDs must be nonnegative integers.")
            if trial_ids.duplicated().any():
                raise ValueError("Duplicate trial identifiers.")
            table["trial"] = trial_ids.astype("int64")
            table = add_saccade_features(table)
            if set(table["task_condition"]) != set(CONDITION_ORDER):
                raise ValueError("At least one of the four experimental conditions is absent.")

            table.insert(0, "subject", subject)
            table["source_file"] = file.name
            parts.append(table)
            log.append({
                "subject": subject,
                "source_file": file.name,
                "sha256": hashlib.sha256(file.read_bytes()).hexdigest(),
                "n_rows_read": n_rows_read,
                "n_blank_rows_removed": int(blank_rows.sum()),
                "n_trials": len(table),
                "n_detected": int(table["saccade_detected"].sum()),
                "n_pretarget": int(table["is_pretarget"].sum()),
                "n_cnv_saccade_ok": int(table["cnv_saccade_ok"].sum()),
            })
        except Exception as error:
            raise ValueError(f"Participant {subject}, file {file.name}: {error}") from error

    trials = pd.concat(parts, ignore_index=True)
    if trials.duplicated(["subject", "trial"]).any():
        raise ValueError("Duplicate participant-by-trial records after concatenation.")
    return trials, pd.DataFrame(log), extra_files

## Summarize trials within each participant

The same summary function is applied to the four experimental conditions and to the two probability contexts. Context-level counts are pooled directly from trials, so the anticipatory probability contrast uses observed detected-trial denominators.

In [7]:
def summarize_saccades(trials, group_columns):
    summary = trials.groupby(group_columns, sort=True, observed=True).agg(
        n_trials=("trial", "size"),
        n_detected=("saccade_detected", "sum"),
        n_anticipatory_lt90=("is_anticipatory", "sum"),
        n_pretarget=("is_pretarget", "sum"),
        n_express=("is_express", "sum"),
        n_regular=("is_regular", "sum"),
        n_incorrect_detected=("is_direction_error", "sum"),
        n_correct_eligible=("rt_ms", "count"),
        mean_srt_correct_ms=("rt_ms", "mean"),
        n_cnv_saccade_ok=("cnv_saccade_ok", "sum"),
    ).reset_index()
    count_columns = [column for column in summary if column.startswith("n_")]
    summary[count_columns] = summary[count_columns].astype("int64")
    summary["n_not_detected"] = summary["n_trials"] - summary["n_detected"]
    denominator = summary["n_detected"].where(summary["n_detected"] > 0)
    summary["anticipatory_pct_detected"] = 100.0 * summary["n_anticipatory_lt90"] / denominator
    summary["error_pct_detected"] = 100.0 * summary["n_incorrect_detected"] / denominator
    summary["detection_pct"] = 100.0 * summary["n_detected"] / summary["n_trials"]

    if not (summary["n_anticipatory_lt90"] + summary["n_express"] + summary["n_regular"]).eq(summary["n_detected"]).all():
        raise AssertionError("Timing categories do not reconcile with detected counts.")
    if not (summary["n_pretarget"] + summary["n_cnv_saccade_ok"]).eq(summary["n_detected"]).all():
        raise AssertionError("Pretarget/posttarget counts do not reconcile.")
    return summary

## Participant-level predictors and contrasts

The eight predictors below are prepared for the behavioral correlations. Means across conditions give equal weight to each available condition value. Missing values are omitted from these means, and the number of contributing conditions is reported in the quality-control table.

| Predictor | Definition | Unit |
|---|---|---|
| `rt_veff80` | Invalid − valid mean response latency in the 80% context | ms |
| `rt_veff50` | Invalid − valid mean response latency in the 50% context | ms |
| `rt_mod` | `rt_veff80 − rt_veff50` | ms |
| `rt_mean` | Equal-weight mean of available condition-level mean response latencies | ms |
| `err_mean` | Equal-weight mean of available condition-level direction-error percentages | % |
| `err_80inv` | Direction-error percentage in the invalid 80% condition | % |
| `antic_mean` | Equal-weight mean of available condition-level anticipatory percentages | % |
| `antic_mod` | Pooled anticipatory percentage at 80% − pooled percentage at 50% | percentage points |

The contrast table also contains the direction-error validity effects, their probability modulation (`error_mod_pp`), and the pooled anticipatory percentages for both contexts. Statistical tests and personality correlations are performed in the statistical-analysis notebook.

In [8]:
def build_behavioral_outputs(participant_condition, participant_probability, participant_ids):
    ids = pd.Index(participant_ids, name="subject")

    def wide(column):
        return participant_condition.pivot(index="subject", columns="condition", values=column).reindex(index=ids, columns=CONDITION_ORDER)

    latency = wide("mean_srt_correct_ms")
    errors = wide("error_pct_detected")
    anticipatory = wide("anticipatory_pct_detected")
    pooled = participant_probability.pivot(
        index="subject", columns="probability", values="anticipatory_pct_detected"
    ).reindex(index=ids, columns=[50, 80])

    features = pd.DataFrame(index=ids)
    features["rt_veff80"] = latency["invalid_80"] - latency["valid_80"]
    features["rt_veff50"] = latency["invalid_50"] - latency["valid_50"]
    features["rt_mod"] = features["rt_veff80"] - features["rt_veff50"]
    features["rt_mean"] = latency.mean(axis=1, skipna=True)
    features["err_mean"] = errors.mean(axis=1, skipna=True)
    features["err_80inv"] = errors["invalid_80"]
    features["antic_mean"] = anticipatory.mean(axis=1, skipna=True)
    features["antic_mod"] = pooled[80] - pooled[50]

    contrasts = features[["rt_veff80", "rt_veff50", "rt_mod", "antic_mod"]].copy()
    contrasts["error_veff80_pp"] = errors["invalid_80"] - errors["valid_80"]
    contrasts["error_veff50_pp"] = errors["invalid_50"] - errors["valid_50"]
    contrasts["error_mod_pp"] = contrasts["error_veff80_pp"] - contrasts["error_veff50_pp"]
    contrasts["anticipatory_80_pct"] = pooled[80]
    contrasts["anticipatory_50_pct"] = pooled[50]

    mean_coverage = pd.DataFrame({
        "n_rt_conditions": latency.notna().sum(axis=1),
        "n_error_conditions": errors.notna().sum(axis=1),
        "n_anticipatory_conditions": anticipatory.notna().sum(axis=1),
    })

    def make_condition_export(frame, prefix, suffix):
        result = frame.reindex(columns=CONDITION_ORDER).rename(
            columns=lambda condition: f"{prefix}_{condition}_{suffix}"
        )
        result.columns.name = None
        return result.rename_axis("subject").reset_index()

    condition_exports = {
        "latency.xlsx": make_condition_export(latency, "latency", "ms"),
        "anticipatory.xlsx": make_condition_export(anticipatory, "anticipatory", "pct"),
        "errors.xlsx": make_condition_export(errors, "errors", "pct"),
    }
    return features.reset_index(), contrasts.reset_index(), mean_coverage.reset_index(), condition_exports

## Table 2: descriptive statistics

For each condition, the mean, sample SD (`ddof=1`), and median are calculated across **participant-level** values. The latency median is the median of participants' mean response latencies. Participants are weighted equally; trials are not pooled across participants for Table 2. The contributing sample size is retained for every cell.

In [9]:
def make_table2(participant_condition):
    measures = {
        "Response latency (ms)": "mean_srt_correct_ms",
        "Anticipatory saccades (%)": "anticipatory_pct_detected",
        "Direction errors (%)": "error_pct_detected",
    }
    rows = []
    for label, column in measures.items():
        for condition in CONDITION_ORDER:
            values = participant_condition.loc[participant_condition["condition"].eq(condition), column].dropna()
            rows.append({
                "measure": label, "condition": condition, "n": len(values),
                "mean": values.mean(), "sd": values.std(ddof=1), "median": values.median(),
            })
    numeric = pd.DataFrame(rows)
    formatted = numeric.assign(
        value=numeric.apply(
            lambda row: f"{row['mean']:.1f} ± {row['sd']:.1f} ({row['median']:.1f})" if row["n"] else "NA",
            axis=1,
        )
    ).pivot(index="measure", columns="condition", values="value")
    formatted = formatted.reindex(index=list(measures), columns=CONDITION_ORDER)
    formatted.columns.name = None
    return numeric, formatted

## Run the study preparation

Set `RUN_STUDY_ANALYSIS = True` to load the reviewed files and calculate the summaries. Every participant in the configured cohort must have a file.

Zero detected counts produce missing percentages. Conditions without a correctly directed express or regular saccade produce a missing response latency. These conditions are reported in the quality-control tables.

In [10]:
study_tables = None
condition_export_tables = None

if RUN_STUDY_ANALYSIS:
    analysis_ids = [subject.strip().upper() for subject in PARTICIPANT_IDS]
    trials, file_qc, extra_files = load_reviewed_saccades(
        SACCADES_DIR, analysis_ids, EXPECTED_N_PARTICIPANTS
    )
    participant_condition = summarize_saccades(
        trials, ["subject", "task_condition", "probability", "validity"]
    ).rename(columns={"task_condition": "condition"})
    participant_probability = summarize_saccades(trials, ["subject", "probability"])
    features, contrasts, mean_coverage, condition_export_tables = build_behavioral_outputs(
        participant_condition, participant_probability, analysis_ids
    )
    table2_numeric, table2_formatted = make_table2(participant_condition)
    missing_metrics = participant_condition.loc[
        participant_condition["n_detected"].eq(0) | participant_condition["n_correct_eligible"].eq(0),
        ["subject", "condition", "n_trials", "n_detected", "n_correct_eligible"],
    ].copy()
    run_summary = pd.DataFrame({
        "item": ["Participants", "Target trials", "Detected saccades", "Pretarget saccades", "Extra files outside cohort", "Conditions with undefined metrics"],
        "value": [trials["subject"].nunique(), len(trials), int(trials["saccade_detected"].sum()), int(trials["is_pretarget"].sum()), len(extra_files), len(missing_metrics)],
    })
    if len(participant_condition) != 4 * EXPECTED_N_PARTICIPANTS:
        raise AssertionError("Unexpected participant-by-condition table size.")
    if len(participant_probability) != 2 * EXPECTED_N_PARTICIPANTS:
        raise AssertionError("Unexpected participant-by-probability table size.")
    if participant_condition["n_trials"].sum() != len(trials):
        raise AssertionError("Condition trial counts do not reconcile.")
    if participant_probability["n_trials"].sum() != len(trials):
        raise AssertionError("Probability trial counts do not reconcile.")

    study_tables = {
        "Trial_annotations": trials,
        "Participant_condition": participant_condition,
        "Participant_probability": participant_probability,
        "Behavioral_features": features,
        "Subject_contrasts": contrasts,
        "Table2_numeric": table2_numeric,
        "Table2_formatted": table2_formatted.reset_index(),
        "File_QC": file_qc,
        "Mean_coverage_QC": mean_coverage,
        "Missing_metrics_QC": missing_metrics,
        "Extra_files_QC": extra_files,
        "Run_summary": run_summary,
        "Software_versions": software_versions,
        "Event_dictionary": EVENT_INFO.reset_index(),
    }
    display(run_summary)
else:
    display(Markdown("**Preview only:** no study files have been read. Set the input directory and `RUN_STUDY_ANALYSIS = True` to compute study outputs."))

**Preview only:** no study files have been read. Set the input directory and `RUN_STUDY_ANALYSIS = True` to compute study outputs.

## Inspect outputs

Inspect the cohort size, trial counts, missing metrics, and condition coverage. Compare Table 2 with the manuscript and check the pooled context counts before using the participant-level predictors in subsequent analyses.

In [11]:
if study_tables is not None:
    display(Markdown("### Input and coverage checks"))
    display(file_qc.head(10))
    if not extra_files.empty:
        display(Markdown("Files outside the specified cohort (not analyzed):"))
        display(extra_files.head(15))
    if not missing_metrics.empty:
        display(Markdown("Conditions with undefined percentages or response latency:"))
        display(missing_metrics.head(15))
    incomplete_means = mean_coverage.loc[
        mean_coverage[["n_rt_conditions", "n_error_conditions", "n_anticipatory_conditions"]].lt(4).any(axis=1)
    ]
    if not incomplete_means.empty:
        display(Markdown("Participants whose condition means use fewer than four nonmissing values:"))
        display(incomplete_means.head(15))
    display(Markdown("### Table 2: mean ± SD (median)"))
    display(table2_formatted)
    display(Markdown("Contributing sample sizes:"))
    display(table2_numeric.pivot(index="measure", columns="condition", values="n").reindex(columns=CONDITION_ORDER))
    display(Markdown("### Participant-level predictors (first 10 rows)"))
    display(features.head(10))

## Export derived tables

Set `EXPORT_OUTPUTS = True` and provide an output directory separate from the reviewed input directory.

| Output | Contents |
|---|---|
| `saccades_summary.xlsx` | Trial-level variables, CNV timing flag, condition/context summaries, eight behavioral predictors, contrasts, Table 2, input hashes, and quality-control tables |
| `latency.xlsx` | Participant-level mean correct response latency for each condition, in milliseconds |
| `anticipatory.xlsx` | Participant-level anticipatory percentage for each condition, among detected saccades |
| `errors.xlsx` | Participant-level direction-error percentage for each condition, among detected saccades |

The three condition workbooks use `subject` as the participant identifier. Their measure columns follow the pattern `<measure>_<validity>_<probability>_<unit>`, for example `latency_valid_80_ms`. All four conditions are exported in `CONDITION_ORDER`.

The `Participant_probability` sheet contains the pooled counts and percentages used for `antic_mod`. These workbooks contain participant-level research data and are separate from the public analysis code.

Set `OVERWRITE_OUTPUTS = True` to replace existing result files.

In [12]:
def export_saccade_tables(tables, condition_exports, output_dir, input_dir, overwrite=False):
    output_dir = Path(output_dir).expanduser()
    if "PATH_TO_" in str(output_dir):
        raise ValueError("Replace the OUTPUT_DIR placeholder before exporting.")
    if output_dir.resolve() == Path(input_dir).expanduser().resolve():
        raise ValueError("Use an output directory separate from the reviewed input directory.")
    targets = [output_dir / "saccades_summary.xlsx"] + [output_dir / name for name in condition_exports]
    existing = [path.name for path in targets if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Outputs already exist: {existing}. Enable OVERWRITE_OUTPUTS to replace them.")

    output_dir.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(targets[0], engine="openpyxl") as writer:
        for sheet, table in tables.items():
            table.to_excel(writer, sheet_name=sheet, index=False)
    for name, table in condition_exports.items():
        table.to_excel(output_dir / name, index=False, engine="openpyxl")
    return pd.DataFrame({"output_file": [path.name for path in targets], "bytes": [path.stat().st_size for path in targets]})

if EXPORT_OUTPUTS:
    if study_tables is None:
        raise RuntimeError("Run the study preparation before exporting.")
    display(export_saccade_tables(study_tables, condition_export_tables, OUTPUT_DIR, SACCADES_DIR, OVERWRITE_OUTPUTS))
else:
    print("Export disabled; no derived files written.")

Export disabled; no derived files written.


## Outputs for subsequent analyses

The `Behavioral_features` sheet contains the eight participant-level predictors. The `Subject_contrasts` sheet contains the within-participant contrasts used for the behavioral comparisons. The `Trial_annotations` sheet includes `cnv_saccade_ok` for matching the saccade timing criterion to EEG epochs.

Before proceeding, inspect:

- the configured cohort and four experimental conditions per participant;
- missing metrics and means based on fewer than four conditions;
- Table 2 and its contributing sample sizes;
- pooled detected and anticipatory counts in each probability context.

The input files must contain the completed visual review. Automatic detections require review before they are used to calculate the study results.